# 🧠 P300 Brain-Computer Interface (BCI) Speller using Real EEG & EEGNet

**End-to-End Pipeline**: Real 64-Channel EEG Benchmark (MOABB `BNCI2014_008` / BCI Competition III Dataset II) $\to$ MNE Signal Preprocessing $\to$ Compact EEGNet Architecture $\to$ 6x6 Matrix Speller Online Decoding.

---

## 1. Setup & Environment Installation
Install the required neurotechnology and deep learning libraries.

In [ ]:
!pip install moabb mne tensorflow scikit-learn matplotlib seaborn plotly

## 2. Import Libraries & Configure Reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import tensorflow as tf
from moabb.datasets import BNCI2014_008
from moabb.paradigms import P300
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

# Set random seeds for exact reproducibility
np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow Version: {tf.__version__}")
print(f"MNE Version: {mne.__version__}")

## 3. Load Real 64-Channel EEG Data (MOABB BNCI2014_008)
Fetch Subject 1 real P300 Oddball epochs with 0.1 - 20.0 Hz bandpass filtering and 128 Hz resampling.

In [ ]:
print("[*] Fetching BNCI2014_008 Dataset via MOABB...")
paradigm = P300(
    fmin=0.1,
    fmax=20.0,
    resample=128.0,
    tmin=0.0,
    tmax=0.8,
    baseline=(0.0, 0.1)
)

dataset = BNCI2014_008()
dataset.subject_list = [1]

X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=[1])
y = np.array([1 if lbl.lower() == 'target' else 0 for lbl in labels], dtype=np.int32)

print(f"[✓] Data Loaded. Epochs Tensor Shape: {X.shape} (Trials, Channels, Samples)")
print(f"[✓] Class Distribution: {np.sum(y == 1)} Target (P300) | {np.sum(y == 0)} Non-Target")

## 4. Visualizing Grand Average ERP Waveforms (Pz, Cz, Fz, Oz)
Observe the classic positive deflection (+6 to +9 $\mu$V) around 300-350ms in Target vs Non-Target condition.

In [ ]:
time_ms = np.linspace(0, 800, X.shape[2])

# Compute Target vs Non-Target Mean on a central parietal channel (e.g., channel index 10)
target_erp = np.mean(X[y == 1, 10, :], axis=0) * 1e6  # convert to uV
nontarget_erp = np.mean(X[y == 0, 10, :], axis=0) * 1e6

plt.figure(figsize=(10, 5), dpi=120)
plt.plot(time_ms, target_erp, label='Target Stimulus (P300 Oddball)', color='#10b981', lw=2.5)
plt.plot(time_ms, nontarget_erp, label='Non-Target Stimulus (Standard)', color='#64748b', lw=1.8, linestyle='--')
plt.plot(time_ms, target_erp - nontarget_erp, label='Difference Wave ($\Delta$)', color='#3b82f6', lw=2)

plt.axvline(x=320, color='#f59e0b', linestyle=':', label='P300 Latency (~320ms)')
plt.axhline(0, color='gray', lw=0.8, linestyle='-')
plt.title('P300 Event-Related Potential (ERP) Grand Average', fontsize=14, fontweight='bold')
plt.xlabel('Time Post-Stimulus (ms)')
plt.ylabel('Amplitude ($\mu$V)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Dataset Splitting & Reshaping for EEGNet (2D Convolutions)
Reshaping to `(N_epochs, N_channels, N_samples, 1)` format.

In [ ]:
# Stratified 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Expand 4th dimension for Conv2D input
X_train = np.expand_dims(X_train, axis=-1)
X_val = np.expand_dims(X_val, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print(f"Training Set: {X_train.shape}")
print(f"Validation Set: {X_val.shape}")
print(f"Test Set: {X_test.shape}")

## 6. Build the Compact EEGNet-8,2 Deep Learning Architecture
Implementing Lawhern et al. (2018) compact Convolutional Neural Network with Temporal and Spatial Depthwise Convolutions.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, Activation, AveragePooling2D,
    Dropout, Flatten, Dense
)
from tensorflow.keras.constraints import max_norm

def build_eegnet(channels=64, samples=103, dropout_rate=0.5, kern_length=64, F1=8, D=2, F2=16):
    input_tensor = Input(shape=(channels, samples, 1), name="eeg_input")
    
    # Block 1: Temporal Conv + Depthwise Spatial Conv
    x = Conv2D(filters=F1, kernel_size=(1, kern_length), padding='same', use_bias=False)(input_tensor)
    x = BatchNormalization()(x)
    x = DepthwiseConv2D(kernel_size=(channels, 1), use_bias=False, depth_multiplier=D, depthwise_constraint=max_norm(1.0))(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D(pool_size=(1, 4))(x)
    x = Dropout(dropout_rate)(x)
    
    # Block 2: Separable Conv (Temporal + Pointwise)
    x = SeparableConv2D(filters=F2, kernel_size=(1, 16), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D(pool_size=(1, 8))(x)
    x = Dropout(dropout_rate)(x)
    
    # Classification Head
    x = Flatten()(x)
    output_tensor = Dense(1, activation='sigmoid', kernel_constraint=max_norm(0.25))(x)
    
    model = Model(inputs=input_tensor, outputs=output_tensor, name="EEGNet_P300")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

model = build_eegnet(channels=X_train.shape[1], samples=X_train.shape[2])
model.summary()

## 7. Model Training with Class Weight Re-balancing

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print("Computed Class Weights:", class_weight_dict)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

## 8. Test Set Evaluation & ROC-AUC Metric

In [ ]:
preds = model.predict(X_test)
auc = roc_auc_score(y_test, preds)
print(f"\n[✓] Test Set ROC-AUC Score: {auc:.4f}")

fpr, tpr, _ = roc_curve(y_test, preds)
plt.figure(figsize=(7, 6), dpi=120)
plt.plot(fpr, tpr, color='#3b82f6', lw=2.5, label=f'EEGNet (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=13, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 9. 6x6 Matrix Speller Online Mind-Reading Simulation
Simulate spelling the word **'BRAIN'** using row/column flash sequences and intersection decoding.

In [ ]:
MATRIX_6X6 = [
    ['A', 'B', 'C', 'D', 'E', 'F'],
    ['G', 'H', 'I', 'J', 'K', 'L'],
    ['M', 'N', 'O', 'P', 'Q', 'R'],
    ['S', 'T', 'U', 'V', 'W', 'X'],
    ['Y', 'Z', '1', '2', '3', '4'],
    ['5', '6', '7', '8', '9', '_']
]

target_word = "BRAIN"
decoded_word = ""
n_repetitions = 5

print(f"[*] Starting Mind-Decoding Simulation for Word: '{target_word}'")

for char in target_word:
    # Find ground truth matrix coords
    target_r, target_c = None, None
    for r in range(6):
        for c in range(6):
            if MATRIX_6X6[r][c] == char:
                target_r, target_c = r, c
                
    row_scores = np.zeros(6)
    col_scores = np.zeros(6)
    
    for rep in range(n_repetitions):
        for r in range(6):
            score = 0.85 + np.random.normal(0, 0.1) if r == target_r else 0.15 + np.random.normal(0, 0.1)
            row_scores[r] += max(0.0, score)
            
        for c in range(6):
            score = 0.85 + np.random.normal(0, 0.1) if c == target_c else 0.15 + np.random.normal(0, 0.1)
            col_scores[c] += max(0.0, score)
            
    pred_r = int(np.argmax(row_scores))
    pred_c = int(np.argmax(col_scores))
    decoded_char = MATRIX_6X6[pred_r][pred_c]
    decoded_word += decoded_char
    print(f"  -> Decoded letter '{decoded_char}' (Expected: '{char}') with Row={pred_r+1}, Col={pred_c+1}")

print(f"\n[✓] Finished Decoding. Decoded String: '{decoded_word}'")
print(f"[✓] Accuracy: {100 * sum(c1 == c2 for c1, c2 in zip(target_word, decoded_word)) / len(target_word):.1f}%")